# 08 — Generalization, regularization, and early stopping

**Level:** Intermediate · **Study time:** 75–90 minutes (including experiments)

**Prerequisites:** Notebooks 05–07.

## What you will be able to do
- Use train, validation, and test sets for different purposes.
- Compare regularized models.
- Restore the best validation checkpoint without aliasing its tensors.

## How to work through this lab
Run cells in order with **Shift+Enter**. Before each result, predict its shape or behavior.
Change one thing, rerun, and explain what changed. Use **Restart Kernel and Run All**
when you want a clean experiment. Lessons use local data and do not depend on earlier kernel state.
Record your observations in the experiment journal near the end.


[Course guide](../README.md) · [Previous: 07](07_classification.ipynb) · [Next: 09](09_cnn_digits.ipynb)


## Step 1 — Prepare a reproducible experiment

A seed makes comparisons easier. Exact numerical results can still differ across devices and PyTorch versions.


In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)  # Small examples often run faster with fewer CPU threads.
device = torch.device("cpu")  # Learn on CPU first; the performance lab covers CUDA.
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True})
print("PyTorch:", torch.__version__, "| Device:", device)


## Step 2 — Create a small noisy training set

With little data, a flexible model may memorize noise. Training adjusts weights; validation selects settings and stopping time; the test set is reserved for the final estimate. Our inputs share a fixed, already bounded scale.


In [ ]:
X = torch.rand(240, 1) * 6 - 3
y = torch.sin(2 * X) + 0.3 * torch.randn_like(X)
tr, va, te = torch.arange(35), torch.arange(35, 130), torch.arange(130, 240)


## Step 3 — Train with validation-based early stopping

Dropout randomly zeros activations during training. Weight decay discourages large weights. `eval()` changes dropout behavior; it does not disable gradients. Use `no_grad()` as well for validation. Deep-copy the best state because `state_dict()` alone holds references to live tensors.


In [ ]:
import copy

def run_experiment(dropout, weight_decay):
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Dropout(dropout),
                          nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.01, weight_decay=weight_decay)
    best_loss, stale, best_epoch = float("inf"), 0, 0
    train_curve, val_curve = [], []
    for epoch in range(500):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss = nn.functional.mse_loss(model(X[tr]), y[tr])
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = nn.functional.mse_loss(model(X[va]), y[va]).item()
        train_curve.append(loss.item()); val_curve.append(val_loss)
        if val_loss < best_loss - 1e-5:
            best_loss, stale, best_epoch = val_loss, 0, epoch
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
        if stale >= 50:
            break
    model.load_state_dict(best_state)
    return {"model": model, "best_val": best_loss, "best_epoch": best_epoch,
            "train_curve": train_curve, "val_curve": val_curve}

runs = {"plain": run_experiment(0., 0.), "regularized": run_experiment(0.1, 0.01)}
for name, run in runs.items():
    print(name, "best validation MSE:", round(run['best_val'], 4), "epoch:", run['best_epoch'])
    plt.plot(run["val_curve"], label=f"{name} validation")
plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.show()


## Step 4 — Select using validation, then evaluate once

Regularization is a hypothesis to test, not a guaranteed improvement. The winning model is chosen using validation results only. If you now tune based on test performance, obtain a fresh final test set for an unbiased assessment.


In [ ]:
winner = min(runs, key=lambda name: runs[name]["best_val"])
best_model = runs[winner]["model"]
best_model.eval()
with torch.no_grad():
    test_mse = nn.functional.mse_loss(best_model(X[te]), y[te]).item()
print("Selected:", winner, "Final test MSE:", test_mse)
assert np.isfinite(test_mse)


## Your turn — practice before opening the solutions

1. Plot the winning run's training and validation curves.
2. Demonstrate that dropout is random in train mode and deterministic in eval mode.
3. Verify the restored model's validation loss equals its recorded best loss. Hint: use eval and no_grad.

Use the empty cell below for your attempt. The worked examples above run as supplied;
you do not need to fill in exercises to use **Run All**. Solutions are below the journal
and their code is initially collapsed in JupyterLab.


In [ ]:
# Your experiment goes here. Add more cells if you need them.


## Experiment journal

| Question | Your notes |
|---|---|
| What did I predict? | |
| What single setting did I change? | |
| What result did I measure? | |
| Why might it have changed? | |
| What would I try next? | |

## Reference solutions
Try the exercises first. Expand the following cell to compare your reasoning.


In [ ]:
selected = runs[winner]
plt.plot(selected['train_curve'], label="train (dropout active if configured)")
plt.plot(selected['val_curve'], label="validation")
plt.legend(); plt.show()
drop = nn.Dropout(0.5)
ones = torch.ones(100)
drop.train(); a, b = drop(ones), drop(ones)
assert not torch.equal(a, b)
drop.eval(); assert torch.equal(drop(ones), drop(ones))
with torch.no_grad():
    restored = nn.functional.mse_loss(best_model(X[va]), y[va]).item()
assert abs(restored - selected['best_val']) < 1e-6


## Before moving on

Explain why a low training loss can coexist with poor validation results. Compare multiple random seeds before making strong claims about small differences.

**Explain it aloud:** What went into the model or operation, what came out, and how did you verify it?

## Official references and further reading
- [Optimization tutorial](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
- [Saving model states](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)


[Course guide](../README.md) · [Previous: 07](07_classification.ipynb) · [Next: 09](09_cnn_digits.ipynb)
